# Практическая работа № 5
## Паттерны State, Strategy и Command

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр  
**Модуль:** 5 — Паттерны: State, Strategy, Command  
**Продолжительность:** 4 ч аудиторно + 4 ч самостоятельно  
**Формат сдачи:** Git-репозиторий + тесты + README + ADR  
**Максимальный балл:** 5

---

## О чём эта работа

После модуля 4 проект «Тайна исчезнувшего дива» умеет обновлять мир с фиксированным шагом, но:
- состояния игры (меню, расследование, диалог, победа) не формализованы;
- поведение Кузи статично;
- действия игрока необратимы.

В этой работе вы добавите три паттерна:
- **State** — состояния игры как классы;
- **Strategy** — стратегии поведения Кузи;
- **Command** — действия с отменой.

## Цели работы

После выполнения вы сможете:
- реализовывать FSM через паттерн State;
- использовать `enter`/`update`/`exit` lifecycle;
- выделять стратегии поведения;
- инкапсулировать действия в команды;
- реализовывать undo/redo через стек;
- тестировать состояния, стратегии и команды.

## План работы

| Шаг | Что делаем | Что изучаем |
|---|---|---|
| 0 | Настройка окружения | ветка Git |
| 1 | Аудит текущего кода | где `if/elif` по состояниям |
| 2 | Проектирование FSM игры | диаграмма состояний |
| 3 | Реализация `GameState` (State) | `enter`, `update`, `exit` |
| 4 | Проектирование стратегий Кузи | `BehaviorStrategy` |
| 5 | Реализация `Command` и `CommandHistory` | `execute`, `undo` |
| 6 | Интеграция в `Game` | State + Command |
| 7 | Юнит-тесты | тесты состояний, стратегий, команд |
| 8 | Документация | ADR, UML |

**Важно:** этот блокнот содержит только задания. Готовые решения — в `practice5_state_strategy_command.ipynb` (образец, не для копирования).

**Для углублённого изучения теории:**
- Nystrom R. «Game Programming Patterns»: [State](https://gameprogrammingpatterns.com/state.html), [Strategy](https://gameprogrammingpatterns.com/strategy.html), [Command](https://gameprogrammingpatterns.com/command.html)
- [Refactoring Guru: State](https://refactoring.guru/design-patterns/state), [Strategy](https://refactoring.guru/design-patterns/strategy), [Command](https://refactoring.guru/design-patterns/command)

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

for pkg in ["pytest", "pycodestyle"]:
    try:
        __import__(pkg)
        print(f"OK: {pkg}")
    except ImportError:
        print(f"MISSING: {pkg} — установите: pip install {pkg}")

---
# Шаг 0. Подготовка

```bash
git checkout main
git checkout -b module5-state-strategy-command
pytest -v
```

Все тесты из модулей 1–4 должны проходить.

---
# Шаг 1. Аудит: где `if/elif` по состояниям

## Зачем этот шаг

Паттерн State заменяет `if/elif` полиморфизмом. Но прежде чем рефакторить, нужно найти все места, где состояния обрабатываются через условия.

## Задание 1.1

Пройдитесь по `Game`, `GameState` и системам. Найдите:
- проверки `if self.state == ...`;
- проверки `if self.mode == ...`;
- флаги `is_menu`, `is_playing`, `in_dialogue`.

Заполните таблицу:

| Место | Условие | Какие состояния | Что должно стать состоянием |
|---|---|---|---|
| ... | ... | ... | ... |

**Сохраните в** `docs/analysis_module5.md`.

## Задание 1.2

Ответьте письменно:
1. Сколько состояний у игры? Перечислите.
2. Какие состояния имеют общее поведение? (кандидаты на иерархию)
3. Какие действия игрока должны быть отменяемыми?

---
# Шаг 2. Проектирование FSM игры

## Задание 2.1. Диаграмма состояний

Нарисуйте диаграмму состояний игры (Mermaid stateDiagram).

Минимум состояний:
- `MenuState` — главное меню.
- `PlayingState` — расследование.
- `VictoryState` — победа.

Опционально:
- `DialogueState` — диалог с персонажем.
- `PauseState` — пауза.

**Сохраните в** `docs/uml_module5.md`.

## Задание 2.2. Ответьте

1. Какие переходы между состояниями?
2. Какие состояния имеют `enter`/`exit` side effects?
3. Нужна ли иерархия? Если да — какая?

---
# Шаг 3. Реализация `GameState` (State)

## Теория

`GameState` — это абстрактный класс с тремя методами:
- `enter(game)` — при входе;
- `update(game, dt)` — каждый кадр;
- `exit(game)` — при выходе.

**Важно:** не путайте `GameState` (состояние игры) с `core/game_state.py` из модуля 3 (хранилище данных). Переименуйте старое хранилище в `WorldState` или `GameSession`.

## Задание 3.1

Создайте `core/game_states.py`:

1. Абстрактный класс `GameState(ABC)` с `enter`, `update`, `exit`.
2. `MenuState` — показывает меню, ждёт выбора.
3. `PlayingState` — основной игровой цикл.
4. `VictoryState` — показывает победу.

**Требования:**
- `enter` выводит заголовок состояния.
- `update` обрабатывает ввод и может переключить состояние через `game.change_state(...)`.
- `exit` очищает состояние.

## Задание 3.2. Коммит

```bash
git add core/game_states.py
git commit -m "Add GameState FSM: Menu, Playing, Victory"
```

---
# Шаг 4. Проектирование стратегий Кузи

## Задание 4.1

Создайте `core/strategies.py`:

1. Протокол `BehaviorStrategy` с методом `decide(div, state) -> str`.
2. `CautiousStrategy` — осторожное поведение.
3. `AggressiveStrategy` — агрессивное поведение.
4. `PlayfulStrategy` — игривое поведение.

**Требования:**
- Каждая стратегия возвращает строку-решение (например, `"scout"`, `"hide"`, `"attack"`).
- `Div` получает стратегию через конструктор.
- Можно менять стратегию на лету через `div.change_strategy(new_strategy)`.

## Задание 4.2. Коммит
```bash
git add core/strategies.py
git commit -m "Add BehaviorStrategy for Div: Cautious, Aggressive, Playful"
```

---
# Шаг 5. Реализация `Command` и `CommandHistory`

## Задание 5.1

Создайте `core/commands.py`:

1. Абстрактный класс `Command` с `execute()` и `undo()`.
2. `MoveCommand(from_loc, to_loc)` — перемещение.
3. `InvestigateCommand(location)` — осмотр.
4. `CommandHistory` — стек выполненных команд.

**Требования:**
- `CommandHistory.execute(cmd)` выполняет и записывает в стек.
- `CommandHistory.undo_last()` отменяет последнюю.
- `undo()` возвращает состояние до выполнения.

## Задание 5.2. Коммит
```bash
git add core/commands.py
git commit -m "Add Command pattern with undo history"
```

---
# Шаг 6. Интеграция в `Game`

## Задание 6.1
Обновите `Game`:
- `Game` хранит `current_state: GameState`;
- `Game.run()` делегирует `update` текущему состоянию;
- `Game.change_state(new_state)` вызывает `exit` старого и `enter` нового;
- действия игрока выполняются через `CommandHistory`.

## Задание 6.2. Проверка
Запустите игру. Убедитесь, что:
- меню → игра → победа работают;
- Кузя меняет поведение;
- можно отменить последний ход.

## Задание 6.3. Коммит
```bash
git add game/
git commit -m "Integrate State, Strategy, Command into Game"
```

---
# Шаг 7. Юнит-тесты

## Задание 7.1. Тесты состояний
`tests/test_game_states.py`:
- `test_enter_called`;
- `test_update_transitions`;
- `test_exit_called`.

## Задание 7.2. Тесты стратегий
`tests/test_strategies.py`:
- `test_cautious_returns_hide`;
- `test_aggressive_returns_attack`;
- `test_change_strategy`.

## Задание 7.3. Тесты команд
`tests/test_commands.py`:
- `test_execute_move`;
- `test_undo_move`;
- `test_history_undo`.

## Задание 7.4. Запуск
```bash
pytest -v
```
Все тесты должны проходить.

---
# Шаг 8. Документация

## Задание 8.1. README
Обновите разделы:
- Состояния игры.
- Стратегии Кузи.
- Команды и отмена.

## Задание 8.2. ADR-005
`docs/adr_005_patterns.md`:
- Контекст: зачем нужны паттерны.
- Решение: State, Strategy, Command.
- Альтернативы: enum + if/elif, таблица переходов.
- Последствия: плюсы и минусы.

## Задание 8.3. UML
Добавьте в `docs/uml_module5.md` диаграммы классов для каждого паттерна.

## Задание 8.4. PEP 8
```bash
pycodestyle core/ game/ tests/ main.py
```
Исправьте замечания.

## Задание 8.5. Финальный коммит
```bash
git add .
git commit -m "Document module 5: State, Strategy, Command"
git checkout main
git merge module5-state-strategy-command
```

---
# Чек-лист
```
[ ] FSM игры спроектирован
[ ] MenuState, PlayingState, VictoryState реализованы
[ ] enter/update/exit работают
[ ] BehaviorStrategy реализован
[ ] Три стратегии Кузи работают
[ ] Command и CommandHistory реализованы
[ ] Undo работает
[ ] Game интегрирован
[ ] Все тесты проходят
[ ] README, ADR, UML обновлены
[ ] PEP 8 без замечаний
```

---
# Критерии оценки

| Критерий | Баллы |
|---|---:|
| State (FSM) | 1.5 |
| Strategy | 1.0 |
| Command + undo | 1.5 |
| Интеграция | 0.5 |
| Тесты | 0.5 |
| **Итого** | **5.0** |

**Штрафы:** −1 за задержку, −2 за плагиат, −1 за отсутствие тестов.

**Бонусы (+1):** иерархические состояния, redo, стратегия с параметрами.

## Что дальше
В модуле 6 — Observer, Factory, Singleton: события, создание объектов, менеджеры.